    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 3 (LS1): Implement a program which (a) given one of the feature models, 
    (b) a user specified value of k, (c) one of the four dimensionality reduction 
    techniques (SVD, NNMF, LDA, k-means) chosen by the user, reports the top-k 
    latent semantics extracted under the selected feature space.

    – Store the latent semantics in a properly named output file

    – List imageID-weight pairs, ordered in decreasing order of weights

In [1]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))

In [2]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 10  latent semantics under  fc  feature space using:  svd


In [3]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(feature_vectors)

latent_semantics = reducer.reduce_features(feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[16.64117176  3.30686378  2.0642993  ...  2.68469761 -0.10995542
  -0.91908201]
 [10.72935793  6.88621642  5.4101054  ...  5.61488896  5.17459503
  -2.84617958]
 [18.86978961  4.88836409  3.85616337 ...  1.43480726  0.15467737
  -1.78788299]
 ...
 [ 9.49211954 -5.15432767 -0.36132027 ...  4.32697608 -1.19752544
   2.01105605]
 [ 8.57595528 -9.00061829 -3.7651228  ...  2.00531902  4.32974302
  -2.40781012]
 [ 5.44301762 -3.40821369 -2.72665178 ...  2.42903829  1.52529653
  -4.49838655]]
Shape:  (4339, 10)


In [4]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'LS1_color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

from utils.database_utils import store

store(reducer, f'LS1_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS1_fc_svd_reducer.pt 



In [5]:
# List imageID-weight pairs, ordered in decreasing order of weights

# We are to showcase which images contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_weight_tuples = list(zip(feature_vectors.keys(), similarity_matrix))

print("Image ID - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMG_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(ID: ", IMG_ID, ", Weight: ", weight[i], end="),\t")

Image ID - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(ID:  800 , Weight:  0.053155404),	(ID:  812 , Weight:  0.052846022),	(ID:  32 , Weight:  0.052470617),	(ID:  230 , Weight:  0.052324735),	(ID:  16 , Weight:  0.052250296),	(ID:  370 , Weight:  0.051881846),	(ID:  10 , Weight:  0.051834393),	(ID:  850 , Weight:  0.051749714),	(ID:  62 , Weight:  0.05172955),	(ID:  834 , Weight:  0.05161436),	(ID:  66 , Weight:  0.05124725),	(ID:  380 , Weight:  0.05102566),	(ID:  808 , Weight:  0.05078927),	(ID:  122 , Weight:  0.050452262),	(ID:  46 , Weight:  0.050447833),	(ID:  186 , Weight:  0.050283164),	(ID:  234 , Weight:  0.04997162),	(ID:  282 , Weight:  0.049920898),	(ID:  372 , Weight:  0.04986296),	(ID:  374 , Weight:  0.04961721),	(ID:  8 , Weight:  0.0495187),	(ID:  780 , Weight:  0.049406476),	(ID:  502 , Weight:  0.04923158),	(ID:  480 , Weight:  0.049029816),	(ID:  336 , Weight:  0.048787314),	(ID:  818 , Weight:  0.048713394),	(